In [0]:
#Data Transformation Layer
from pyspark.sql import functions as F
import re

dbutils.widgets.text("catalog", "insurance_project")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

for value in [catalog, bronze_schema, silver_schema]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError(f"Invalid catalog/schema name: {value}")

primary_keys = {
    "branches": "branch_id",
    "insurance_products": "product_id",
    "agents": "agent_id",
    "customers": "customer_id",
    "policies": "policy_id",
    "premium_payments": "payment_id",
    "beneficiaries": "beneficiary_id",
    "claims": "claim_id",
    "claim_assessments": "assessment_id",
    "claim_settlements": "settlement_id"
}

money_columns = {
    "annual_income",
    "coverage_limit",
    "annual_premium",
    "deductible_amount",
    "sum_insured",
    "premium_due",
    "amount_paid",
    "outstanding_amount",
    "claimed_amount",
    "approved_amount",
    "paid_amount"
}

percentage_columns = {
    "commission_rate_percent",
    "allocation_percent"
}

boolean_columns = {
    "is_active",
    "is_primary",
    "fraud_flag"
}

def column_type(column_name):
    if column_name in money_columns:
        return "DECIMAL(18,2)"
    if column_name in percentage_columns:
        return "DECIMAL(5,2)"
    if column_name in boolean_columns:
        return "BOOLEAN"
    if column_name == "policy_term_days":
        return "INT"
    if column_name == "date_of_birth":
        return "DATE"
    if column_name.endswith("_date"):
        return "DATE"
    if column_name in {"created_at", "updated_at"}:
        return "TIMESTAMP"
    return "STRING"


In [0]:
cleaned = {}

for table_name, primary_key in primary_keys.items():
    raw = spark.table(
        f"{catalog}.{bronze_schema}.{table_name}"
    )

    business_columns = [
        c for c in raw.columns if not c.startswith("_")
    ]

    metadata_columns = [
        c for c in raw.columns if c.startswith("_")
    ]

    # Trim strings and convert empty strings to null.
    normalized = raw.select(
        *[
            F.when(
                F.trim(F.col(c)) == "",
                F.lit(None)
            )
            .otherwise(F.trim(F.col(c)))
            .alias(c)
            for c in business_columns
        ],
        *[F.col(c) for c in metadata_columns]
    )

    conversions = {
        c: F.expr(
            f"try_cast(`{c}` AS {column_type(c)})"
        )
        for c in business_columns
    }

    # A populated source value must not become null after casting.
    cast_failure = F.lit(False)

    for c in business_columns:
        cast_failure = cast_failure | (
            F.col(c).isNotNull()
            & conversions[c].isNull()
        )

    if normalized.filter(cast_failure).limit(1).count():
        raise ValueError(
            f"{table_name}: one or more values cannot be cast"
        )

    typed = normalized.select(
        *[
            conversions[c].alias(c)
            for c in business_columns
        ],
        *[F.col(c) for c in metadata_columns]
    )

    if "email" in typed.columns:
        typed = typed.withColumn(
            "email", F.lower(F.col("email"))
        )

    if "currency" in typed.columns:
        typed = typed.withColumn(
            "currency", F.upper(F.col("currency"))
        )

    if typed.filter(
        F.col(primary_key).isNull()
    ).limit(1).count():
        raise ValueError(
            f"{table_name}: null primary key"
        )

    duplicate_keys = (
        typed.groupBy(primary_key)
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicate_keys.limit(1).count():
        raise ValueError(
            f"{table_name}: duplicate primary key"
        )

    for c in money_columns.intersection(typed.columns):
        if typed.filter(F.col(c) < 0).limit(1).count():
            raise ValueError(
                f"{table_name}.{c}: negative amount"
            )

    cleaned[table_name] = typed

    print(f"Prepared: {table_name}")


In [0]:
parent_tables = {
    "branch_id": "branches",
    "product_id": "insurance_products",
    "agent_id": "agents",
    "customer_id": "customers",
    "policy_id": "policies",
    "claim_id": "claims"
}

for table_name, df in cleaned.items():
    own_primary_key = primary_keys[table_name]

    for foreign_key, parent_table in parent_tables.items():
        if (
            foreign_key in df.columns
            and foreign_key != own_primary_key
        ):
            parent_ids = (
                cleaned[parent_table]
                .select(foreign_key)
                .distinct()
            )

            orphans = df.join(
                parent_ids,
                on=foreign_key,
                how="left_anti"
            )

            if orphans.limit(1).count():
                raise ValueError(
                    f"{table_name}.{foreign_key}: "
                    f"missing reference in {parent_table}"
                )

# This snapshot has exactly one assessment per claim.
assessment_duplicates = (
    cleaned["claim_assessments"]
    .groupBy("claim_id")
    .count()
    .filter(F.col("count") > 1)
)

if assessment_duplicates.limit(1).count():
    raise ValueError("Multiple assessments found for one claim")

missing_assessments = (
    cleaned["claims"]
    .select("claim_id")
    .join(
        cleaned["claim_assessments"].select("claim_id"),
        on="claim_id",
        how="left_anti"
    )
)

if missing_assessments.limit(1).count():
    raise ValueError("A claim has no assessment")

# Customer must match the customer on the referenced policy.
for child_name in [
    "claims", "premium_payments", "beneficiaries"
]:
    child = cleaned[child_name].alias("c")
    policy = cleaned["policies"].alias("p")

    mismatch = (
        child.join(
            policy,
            F.col("c.policy_id") == F.col("p.policy_id"),
            "inner"
        )
        .filter(
            ~F.col("c.customer_id").eqNullSafe(
                F.col("p.customer_id")
            )
        )
    )

    if mismatch.limit(1).count():
        raise ValueError(
            f"{child_name}: customer does not match policy"
        )

# Incidents must be inside the policy coverage dates.
claim_policy = (
    cleaned["claims"].alias("c")
    .join(
        cleaned["policies"].alias("p"),
        F.col("c.policy_id") == F.col("p.policy_id"),
        "inner"
    )
)

invalid_dates = claim_policy.filter(
    F.col("c.incident_date").isNull()
    | F.col("p.start_date").isNull()
    | F.col("p.end_date").isNull()
    | (F.col("c.incident_date") < F.col("p.start_date"))
    | (F.col("c.incident_date") > F.col("p.end_date"))
    | (F.col("c.claim_date") < F.col("c.incident_date"))
)

if invalid_dates.limit(1).count():
    raise ValueError("Invalid claim or coverage dates")

# Settlement policy must match the claim's policy.
invalid_settlements = (
    cleaned["claim_settlements"].alias("s")
    .join(
        cleaned["claims"].alias("c"),
        F.col("s.claim_id") == F.col("c.claim_id"),
        "inner"
    )
    .filter(
        ~F.col("s.policy_id").eqNullSafe(F.col("c.policy_id"))
        | (F.col("c.claim_status") != "Settled")
    )
)

if invalid_settlements.limit(1).count():
    raise ValueError("Invalid settlement-to-claim relationship")

print("Relationship checks passed")


In [0]:
paid_by_claim = (
    cleaned["claim_settlements"]
    .groupBy("claim_id")
    .agg(F.sum("paid_amount").alias("total_paid"))
)

settled_claims = (
    cleaned["claims"]
    .filter(F.col("claim_status") == "Settled")
    .select("claim_id")
    .join(
        cleaned["claim_assessments"]
        .select("claim_id", "approved_amount"),
        on="claim_id",
        how="left"
    )
    .join(
        paid_by_claim,
        on="claim_id",
        how="left"
    )
)

bad_totals = settled_claims.filter(
    F.col("approved_amount").isNull()
    | F.col("total_paid").isNull()
    | (F.col("approved_amount") != F.col("total_paid"))
)

if bad_totals.limit(1).count():
    raise ValueError(
        "Settled claim payments do not equal approved amounts"
    )

audit_rows = []

for table_name, df in cleaned.items():
    target_table = f"{catalog}.{silver_schema}.{table_name}"

    output_df = df.withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )

    (
        output_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    audit_rows.append(
        (table_name, output_df.count(), target_table)
    )

display(
    spark.createDataFrame(
        audit_rows,
        ["table_name", "row_count", "target_table"]
    )
)

spark.table(
    f"{catalog}.{silver_schema}.policies"
).printSchema()
